# HPO SolarSystemLander

Elise HPO (8D) from `hpo/designs/design.md`.

## Set up Colab

In [1]:
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.colab import setup_colab

COLAB = setup_colab()

## Set up HPO

In [ ]:
import torch
from hpo.colab import prepare_storage

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OBSERVATION_MODE = "8d"
STUDY_SERIES_NAME = f"solar_system_lander_{OBSERVATION_MODE}_elise"
STUDY_SERIES_STORAGE = prepare_storage(COLAB, STUDY_SERIES_NAME)

In [ ]:
from hpo.objective import ObjectiveConfig
from hpo.solar_system_lander.environment import EnvFactory

ENV_FACTORY = EnvFactory(OBSERVATION_MODE)
OBJECTIVE_CFG = ObjectiveConfig(
    environment_factory=ENV_FACTORY,
    num_envs=20,
    device=device,
    eval_episodes=10,
)

## Optimize HPs iteratively

### Define baseline

Define incumbent HPs

TODO: What's the best checkpoint, i.e. incumbent model?

#### Alternative 1: Set manually

In [ ]:
from hpo.params import HP
from hpo.study import Baseline

BASELINE_PARAMS = {
    HP.LEARNING_RATE: 0.0022727854024196057,
    HP.BATCH_SIZE: 512,
    HP.EPS_END: 0.047716002108220544,
    HP.EPS_DECAY: 43_214,
    HP.GAMMA: 0.99,
    HP.TAU: 0.005,
    HP.LEARNING_STARTS: 2_500,
    HP.OPTIMIZE_EVERY: 2,
    HP.REPLAY_MEMORY_CAPACITY: 400_000,
    HP.NUM_EPISODES: 500,
}

baseline = Baseline(params=BASELINE_PARAMS, score=118.09342221642783)

#### Alternative 2: Read from DB

In [ ]:
from hpo.study import Baseline

baseline = Baseline.from_database("runs/previous.db", "s4")
params = baseline.params
score = baseline.score

### Create study runner

In [ ]:
from hpo.evaluation.dashboard import Dashboard
from hpo.study import StudyRunner

runner = StudyRunner(
    database_path=lambda _name: STUDY_SERIES_STORAGE.database_path,
    objective_cfg=OBJECTIVE_CFG,
    baseline=baseline,
    reporter=Dashboard(),
    study_attrs=ENV_FACTORY.metadata(),
    robust_candidates=5,
    extra_seeds=(1001, 1002, 1003, 1004),
    sync_fn=STUDY_SERIES_STORAGE.backup,
)

### Run studies

In [ ]:
def suggest_params(trial, _incumbent_params):
    trial.suggest_categorical(HP.NUM_EPISODES, [500, 750, 1_000])

runner.run("s1_flight_hours", suggest_params, 25)

In [ ]:
def suggest_params(trial, _incumbent_params):
    trial.suggest_float(HP.EPS_END, 0.03, 0.07)
    trial.suggest_int(HP.EPS_DECAY, 30_000, 150_000, log=True)

runner.run("s2_exploration", suggest_params, 25)

## Reload studies

In [ ]:
# Run only after the runtime environment has been reset.
import optuna

def load_study(name):
    return optuna.load_study(study_name=name, storage=f"sqlite:///{STUDY_SERIES_STORAGE.database_path}")

study1 = load_study("s1_flight_hours")
study2 = load_study("s2_exploration")

## Analyze results

In [ ]:
import pandas as pd

studies = [study1, study2]
labels = ["S1", "S2"]

rows = []
for label, study in zip(labels, studies):
    rows.append({
        "study": label,
        "score": study.user_attrs["incumbent_score"],
    })

display(pd.DataFrame(rows).set_index("study").style.format("{:.1f}"))

### Is the lander still learning at the end?

In [ ]:
from pathlib import Path

import optuna
import pandas as pd
import plotly.express as px

tmp_dir = Path("hpo/tmp")
if not tmp_dir.exists():
    tmp_dir = Path("../tmp")
database_path = (tmp_dir / f"{STUDY_SERIES_NAME}.db").resolve()
study = optuna.load_study(
    study_name="s2_exploration",
    storage=f"sqlite:///{database_path}",
)
params = study.user_attrs.get("robust_best_params")
trials = [
    trial for trial in study.trials
    if trial.state.name == "COMPLETE" and trial.params == params
]
#trial = max(trials, key=lambda trial: trial.value) if trials else study.best_trial
trial = study.best_trial

returns = trial.user_attrs["training_curve"]["episode_returns"]
window = min(100, len(returns) // 2)
curve = pd.DataFrame({"Episode return": returns})
curve[f"Mean ({window} episodes)"] = curve["Episode return"].rolling(window).mean()

display(px.line(curve, labels={"index": "Episode", "value": "Return"}))

previous_mean = curve["Episode return"].iloc[-2 * window:-window].mean()
final_mean = curve["Episode return"].iloc[-window:].mean()
print(f"Previous {window} episodes: {previous_mean:.1f}")
print(f"Final {window} episodes:    {final_mean:.1f}")
print(f"Change:                    {final_mean - previous_mean:+.1f}")

### Should the trials have trained longer?

In [ ]:
import numpy as np

def estimate_alpha(returns, window):
    returns = np.asarray(returns)
    means = [
        returns[-3 * window:-2 * window].mean(),
        returns[-2 * window:-window].mean(),
        returns[-window:].mean(),
    ]
    previous_gain = means[1] - means[0]
    latest_gain = means[2] - means[1]
    return np.nan if np.isclose(previous_gain, 0) else latest_gain / previous_gain

In [ ]:
import numpy as np
import plotly.graph_objects as go

rows = []
for trial in study.trials:
    if trial.state.name != "COMPLETE" or "training_curve" not in trial.user_attrs:
        continue

    returns = np.asarray(trial.user_attrs["training_curve"]["episode_returns"])
    window = min(50, len(returns) // 3)
    previous = returns[-2 * window:-window]
    final = returns[-window:]
    improvement = final.mean() - previous.mean()
    error = 1.96 * np.sqrt(
        previous.var(ddof=1) / window + final.var(ddof=1) / window
    )
    alpha = estimate_alpha(returns, window)
    rows.append((
        trial.number,
        improvement,
        error,
        improvement > error,
        alpha,
    ))

figure = go.Figure(go.Bar(
    x=[row[0] for row in rows],
    y=[row[1] for row in rows],
    error_y={"type": "data", "array": [row[2] for row in rows]},
    marker_color=["green" if row[3] else "gray" for row in rows],
    text=[f"α={row[4]:.1f}" if np.isfinite(row[4]) else "α=n/a" for row in rows],
    textposition="outside",
    textfont={"size": 14},
))
figure.add_hline(y=0, line_color="black")
figure.update_layout(
    xaxis_title="Trial",
    yaxis_title="Change in mean episode return",
    title=(
        "Green: evidence that longer training may help"
        "<br><sup>Balken: Differenz der Mittelwerte der letzten zwei Fenster</sup>"
    ),
)
figure.show()

print("window = " + str(window))